# Small-book quote-or-refuse — finetune (2026-10-13, v7 — x2d)

Before running: **Runtime -> Change runtime type -> A100 if offered, else L4** (High-RAM not needed).

One session trains **one candidate** (Qwen2.5-3B base, x2d; pre-registration:
`x2d-plan-2026-10-14.md`). When the picker opens, upload exactly one file:
`quote-refuse-2026-10-13.train-x2d.jsonl`.
Do not upload the bench file.

The single slot exports q4_k_m + q8_0, plus one LoRA zip and one run-log JSON.
The q8 zip is the quant A/B from the pre-registered gate
(`x2-trial-2026-10-14.md`, item 5): a wart that vanishes at q8_0 is a
quantization verdict, not a training verdict.

Watch the two **STOP-CHECK** lines after training: the reply should be
one quote + one gloss line, then silence. A SUSPECT verdict means tell
the agent before trusting that zip.

If a cell errors, copy the red text back to the agent; library drift happens.

v7: single slot — no seed control. Same toolchain pin as v5
(`unsloth==2026.10.5`, `transformers==5.17.0`, fallback to latest) and the
versions actually loaded are printed. The expected row count comes from
EXPECTED_X2D (patched from
`data/finetune/quote-refuse-2026-10-13.stats-x2d.json` after the build);
it is asserted on load and again after responses-only prep, as the
truncation canary. Training is epoch-based (6 epochs) so per-row exposure
matches x2's level; the dataset size changed.

Env trap (2026-10-14): a fresh Unsloth build defaulted the trainer to
max_length=1024 and silently dropped 152/167 rows (the assistant marker
was truncated away). The notebook passes SFTConfig(max_length=max_seq_length)
and adds the canary assert; watch the trainer banner — 'Num examples' must
match EXPECTED_X2D.


In [ ]:
!pip install -q "unsloth==2026.10.5" "transformers==5.17.0" || pip install -q unsloth

In [ ]:
import importlib

for pkg in ["unsloth", "torch", "transformers", "trl", "peft"]:
    try:
        mod = importlib.import_module(pkg)
        print(pkg, getattr(mod, "__version__", "(no __version__)"))
    except Exception as exc:
        print(pkg, "import failed:", repr(exc))

In [ ]:
import gc
import hashlib
import json
import os
import re
import torch

EXPECTED_X2D = 328  # patched 2026-10-14 from stats-x2d (train rows: 274 answer + 54 refusal)
assert EXPECTED_X2D, "EXPECTED_X2D not patched from stats-x2d — do not run"

RUNS = [
    {"model": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
     "export": "smallbook_x2d",
     "data": "quote-refuse-2026-10-13.train-x2d.jsonl",
     "expected": EXPECTED_X2D,
     "seed": 3407},  # x2d single slot (2026-10-14): form-augmented set, one run
    # If a model tag 404s: send the red text to the agent; do not invent tags.
]

REFUSAL = "Not in this book."


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


missing = [r["data"] for r in RUNS if not os.path.exists(r["data"])]
if missing:
    from google.colab import files
    files.upload()  # select quote-refuse-2026-10-13.train-x2d.jsonl
for r in RUNS:
    print(r["data"], "present:", os.path.exists(r["data"]))
assert all(os.path.exists(r["data"]) for r in RUNS), "upload quote-refuse-2026-10-13.train-x2d.jsonl"


In [ ]:
def train_one(model_name, export_name, data_file, expected, seed, q8=False):
    gc.collect()
    torch.cuda.empty_cache()

    from datasets import load_dataset
    raw = load_dataset("json", data_files=data_file, split="train")
    print(len(raw), "cards loaded from", data_file)
    assert len(raw) == expected, (
        "row count mismatch: %s has %d cards, expected %d" % (data_file, len(raw), expected))
    data_sha = sha256_file(data_file)
    refusals = sum(1 for i in range(len(raw))
                   if raw[i]["messages"][-1]["content"].strip() == REFUSAL)
    print("data sha256:", data_sha)
    print("refusals in file:", refusals, "of", len(raw))

    from unsloth import FastLanguageModel
    from unsloth.chat_templates import train_on_responses_only
    from trl import SFTConfig, SFTTrainer
    from transformers import TrainingArguments

    max_seq_length = 4096
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=seed,
    )

    dataset = raw.map(lambda row: {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)})

    # Truncation guard (2026-10-14: a fresh Unsloth build defaulted
    # max_length=1024 and silently dropped 152/167 rows whose assistant
    # marker was cut away; those rows teach "never stop").
    lens = sorted(len(tokenizer(d["text"])["input_ids"]) for d in dataset)
    print("token lens p50/p90/max:", lens[len(lens) // 2],
          lens[int(.9 * len(lens)) - 1], lens[-1])
    print("rows over max_length:", sum(1 for x in lens if x > max_seq_length))
    assert lens[-1] <= max_seq_length, "truncation would cut the tail (EOS)"

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        args=SFTConfig(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=6,  # 6 epochs keeps per-row exposure at x2's level; the dataset size changed
            max_length=max_seq_length,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=seed,  # per-run seed (RUNS); SFTConfig subclasses TrainingArguments
            output_dir="outputs",
            report_to="none",
        ),
    )
    # Train on the model's ANSWERS only — mask the system prompt and the
    # passages. v1 trained on the whole text and never learned to stop.
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    # Truncation canary: a bad max_length silently drops rows whose
    # assistant marker was cut (152/167 were lost on 2026-10-14).
    print("rows after responses-only prep:", data_file, len(trainer.train_dataset), "of", expected)
    assert len(trainer.train_dataset) == expected, (
        "truncation canary: %s kept only %d of %d rows through dataset prep" % (
            data_file, len(trainer.train_dataset), expected))

    # Mask/EOS spot-check (2026-10-14 review P0-3): proves system+passage
    # tokens are masked and the answer's end token is supervised. An
    # unsupervised EOS is how "never stop" gets taught.
    sup_count = None
    try:
        batch = trainer.data_collator([trainer.train_dataset[0]])
        keep = [t.item() for t, l in zip(batch["input_ids"][0], batch["labels"][0]) if l != -100]
        sup = tokenizer.decode(keep)
        sup_count = len(keep)
        print("supervised tokens:", len(keep), "of", batch["labels"].shape[1])
        print("supervised tail:", repr(sup[-120:]))
        assert sup.rstrip().endswith("<|im_end|>"), "EOS/im_end not supervised - do not train"
    except AssertionError:
        raise
    except Exception as exc:
        print("MASK CHECK could not run (API drift?) - tell the agent:", repr(exc))

    print("config:", trainer.args.max_length, trainer.args.max_steps,
          getattr(trainer.args, "packing", None), trainer.args.seed,
          trainer.args.lr_scheduler_type, trainer.args.learning_rate)
    trainer.train()

    # Stop check BEFORE exporting: one answer row + one refusal row (2026-10-14
    # review P0-3). A reply is sane when it matches the exact 2-line format or
    # the exact refusal, ends with <|im_end|>, and stays under 250 tokens.
    FastLanguageModel.for_inference(model)
    stop_checks = {}

    def stop_check(idx, label):
        prompt_part = dataset[idx]["text"].split("<|im_start|>assistant\n")[0] + "<|im_start|>assistant\n"
        inputs = tokenizer([prompt_part], return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
        reply_ids = out[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(reply_ids, skip_special_tokens=False)
        ok = (re.match(r'^(\[Ch\. \d+\] ".*?"\n\S.*|Not in this book\.)', text)
              and text.rstrip().endswith("<|im_end|>")
              and reply_ids.shape[0] < 250)
        print("STOP-CHECK", label, "reply:\n", text[:500])
        print("STOP-CHECK", label, "verdict:", "sane" if ok else "SUSPECT - tell the agent")
        stop_checks[label] = {"verdict": "sane" if ok else "SUSPECT", "reply": text[:500]}

    answer_idx = next(i for i in range(len(dataset))
                      if raw[i]["messages"][-1]["content"].startswith("[Ch."))
    refusal_idx = next(i for i in range(len(dataset))
                       if raw[i]["messages"][-1]["content"].strip() == REFUSAL)
    stop_check(answer_idx, "answer-row")
    stop_check(refusal_idx, "refusal-row")

    # Save the LoRA adapter before export: a future quant can be re-made
    # without retraining (2026-10-14 review P0-3/P1-6; x2 lost its adapter).
    model.save_pretrained(export_name + "_lora")

    model.save_pretrained_gguf(export_name, tokenizer, quantization_method="q4_k_m")
    import glob, shutil
    from google.colab import files as colab_files
    path = sorted(glob.glob(export_name + "_gguf/*.gguf"))[0]
    print("gguf:", path)
    shutil.make_archive(export_name + "_gguf", "zip", root_dir=path.rsplit("/", 1)[0])
    q4_sha = sha256_file(path)
    q4_zip_sha = sha256_file(export_name + "_gguf.zip")
    print("gguf sha256:", q4_sha)
    print("zip sha256:", q4_zip_sha)
    shutil.make_archive(export_name + "_lora", "zip", root_dir=export_name + "_lora")
    colab_files.download(export_name + "_lora.zip")
    colab_files.download(export_name + "_gguf.zip")
    print("done:", export_name)

    q8_sha = None
    q8_zip_sha = None
    if q8:
        # Quant A/B, pre-registered gate item 5 (x2-trial-2026-10-14.md) and
        # the 2026-10-14 q4-vs-q8 design: same harnesses, same draw counts,
        # per-item diff; a wart vanishes only if q8 is clean across all draws.
        # A failure here must not kill the session — the q4 zip above is safe.
        try:
            model.save_pretrained_gguf(export_name + "_q8", tokenizer,
                                       quantization_method="q8_0")
            q8_path = sorted(glob.glob(export_name + "_q8_gguf/*.gguf"))[0]
            print("q8 gguf:", q8_path)
            shutil.make_archive(export_name + "_q8_gguf", "zip",
                                root_dir=q8_path.rsplit("/", 1)[0])
            q8_sha = sha256_file(q8_path)
            q8_zip_sha = sha256_file(export_name + "_q8_gguf.zip")
            print("q8 gguf sha256:", q8_sha)
            print("q8 zip sha256:", q8_zip_sha)
            colab_files.download(export_name + "_q8_gguf.zip")
            print("done:", export_name + "_q8_gguf.zip")
        except Exception as exc:
            print("WARNING: q8_0 export/zip/download failed; the q4 zip is",
                  "already safe, session continues:", repr(exc))

    # Run log (2026-10-14 review P1-6): one JSON per run, downloaded with the
    # zips — the comparability record the x2 run is missing.
    run_log = {
        "export": export_name,
        "model": model_name,
        "data": data_file,
        "data_sha256": data_sha,
        "rows": len(raw),
        "refusals": refusals,
        "gpu": torch.cuda.get_device_name(0),
        "versions": {pkg: getattr(importlib.import_module(pkg), "__version__", None)
                     for pkg in ["unsloth", "torch", "transformers", "trl", "peft"]},
        "seed": seed,
        "args": json.loads(trainer.args.to_json_string()),
        "packing": getattr(trainer.args, "packing", None),
        "token_lens": {"p50": lens[len(lens) // 2], "p90": lens[int(.9 * len(lens)) - 1],
                       "max": lens[-1],
                       "over_max_length": sum(1 for x in lens if x > max_seq_length)},
        "supervised_tokens": sup_count,
        "stop_checks": stop_checks,
        "loss_history": trainer.state.log_history,
        "gguf_sha256": {"q4": q4_sha, "q8": q8_sha},
        "zip_sha256": {"q4": q4_zip_sha, "q8": q8_zip_sha},
    }
    with open("run-log-%s.json" % export_name, "w") as handle:
        json.dump(run_log, handle, indent=2, default=str)
    colab_files.download("run-log-%s.json" % export_name)
    print("run log done:", export_name)


In [ ]:
# Run A: x2d — single slot, seed 3407 (A100 if offered, else L4).
# Exports q4_k_m + q8_0: the q8 zip is the quant A/B from the
# pre-registered gate (x2-trial-2026-10-14.md).
train_one(RUNS[0]["model"], RUNS[0]["export"], RUNS[0]["data"],
          RUNS[0]["expected"], RUNS[0]["seed"], q8=True)
